# ELECTRE
---
_**Él**imination **E**t **C**hoix **T**raduisant la **Re**alité_

In [1]:
import numpy as np
import pandas as pd

from itertools import combinations

## 1. Introduction
---
**ELECTRE** is a family of methods based on **outranking**: the assertion that one alternative is at least as good as another. Here, we implement **ELECTRE I**, which supports **choice** by combining a sufficient coalition of supporting criteria with a test against excessive disadvantages.

The result is a **kernel**, a subset of alternatives retained for consideration. ELECTRE I does not produce a complete numerical ranking.

### 1.1. Applications
---
The _ELECTRE I method_ supports choices where a large disadvantage should not be offset by advantages elsewhere, such as:

- Facility location
- Equipment procurement
- Transport infrastructure selection
- Environmental project selection

Consider **4 sites** for a distribution center. All satisfy the project's legal and capacity requirements. Compare **annual cost** (BRL million/year), **delivery time** (minutes), **capacity** (orders/day), and expected **closure days** (days/year due to flooding or access disruption).

The following data and decision policies are **illustrative**, representing a realistic planning exercise rather than measurements of named sites.

In [2]:
df = pd.DataFrame({
    "annual cost": [4.8, 5.4, 5.9, 6.2],
    "delivery time": [35, 35, 28, 42],
    "capacity": [1800, 1800, 1600, 2100],
    "closure days": [18, 2, 4, 6]
})

df.index = ["Riverside", "North", "East", "West"]

df.style.background_gradient()

,annual cost,delivery time,capacity,closure days
Riverside,4.800000,35,1800,18
North,5.400000,35,1800,2
East,5.900000,28,1600,4
West,6.200000,42,2100,6


## 2. Method steps
---

### 2.1. Objectives and weights
---
Define each criterion's **objective** and normalized **weight**. Weights represent the voting power of criteria in a supporting coalition, rather than compensation rates between performances.

The planning team gives the greatest weight to continuity of service; capacity and cost follow. It also sets the **concordance threshold** to $0.70$ and the **discordance threshold** to $0.35$ before comparing the sites. Section 2.5 interprets these parameters on the common scales.

In [3]:
OBJECTIVES = pd.Series(["min", "min", "max", "min"], index=df.columns)
WEIGHTS = pd.Series([0.25, 0.20, 0.25, 0.30], index=df.columns)

C_THRESHOLD = 0.70
D_THRESHOLD = 0.35
TOLERANCE = 1e-12

assert (WEIGHTS > 0).all() and np.isclose(WEIGHTS.sum(), 1)

pd.DataFrame({"objective": OBJECTIVES, "weight": WEIGHTS})

,objective,weight
annual cost,min,0.25
delivery time,min,0.20
capacity,max,0.25
closure days,min,0.30


### 2.2. Common performance scales
---
The **common-scale formulation** of ELECTRE I in Figueira, Mousseau and Roy (2005, section 3.1.1) compares disadvantages on numerical scales with identical ranges. Here, the team agrees on fixed **lower** and **upper planning bounds** and linear coding into $[0,1]$:

$$ \large
g_j(a_i) =
\begin{cases}
\dfrac{x_{ij}-L_j}{U_j-L_j}, & \text{if criterion } j \text{ is maximized}, \\
\dfrac{U_j-x_{ij}}{U_j-L_j}, & \text{if criterion } j \text{ is minimized}.
\end{cases}
$$

Larger values are always preferred. The bounds below define the team's assessment ranges; they are not estimated from the current alternatives. This coding assumes that equal fractions of these ranges express comparable disadvantages. Different coding requires reviewing the discordance threshold.

In [4]:
BOUNDS = pd.DataFrame({
    "lower": [4.0, 20, 1200, 0],
    "upper": [8.0, 60, 2400, 20]
}, index=df.columns)

BOUNDS

,lower,upper
annual cost,4.0,8.0
delivery time,20.0,60.0
capacity,1200.0,2400.0
closure days,0.0,20.0


In [5]:
assert (BOUNDS["upper"] > BOUNDS["lower"]).all()
assert df.ge(BOUNDS["lower"]).all().all()
assert df.le(BOUNDS["upper"]).all().all()

df_n = (df - BOUNDS["lower"])/(BOUNDS["upper"] - BOUNDS["lower"])

for col in df_n.columns:
    if OBJECTIVES[col] == "min":
        df_n[col] = 1 - df_n[col]

df_n.style.background_gradient(vmin=0, vmax=1)

,annual cost,delivery time,capacity,closure days
Riverside,0.800000,0.625000,0.500000,0.100000
North,0.650000,0.625000,0.500000,0.900000
East,0.525000,0.800000,0.333333,0.800000
West,0.450000,0.450000,0.750000,0.700000


### 2.3. Concordance sets and index
---
For each ordered pair $(a,b)$, identify the criteria supporting $a$ being **at least as good as** $b$:

$$ \large
J^+(a,b)=\{j:g_j(a)\geq g_j(b)\}, \qquad
c(a,b)=\sum_{j\in J^+(a,b)}w_j
$$

Equal performances support both directions. Rows refer to $a$ and columns to $b$ throughout the pairwise matrices.

In [6]:
CONCORDANCE_SETS = {}
DISCORDANCE_SETS = {}
df_c = pd.DataFrame(0.0, index=df.index, columns=df.index)

for a in df.index:
    for b in df.index:
        support = df_n.loc[a] >= df_n.loc[b]
        CONCORDANCE_SETS[a, b] = list(df.columns[support])
        DISCORDANCE_SETS[a, b] = list(df.columns[~support])
        df_c.loc[a, b] = WEIGHTS[support].sum()

df_sets = pd.DataFrame([
    {
        "comparison": f'{a} -> {b}',
        "concordant criteria": ", ".join(CONCORDANCE_SETS[a, b]),
        "discordant criteria": ", ".join(DISCORDANCE_SETS[a, b])
    }
    for a in df.index for b in df.index if a != b
]).set_index("comparison")

df_sets

,concordant criteria,discordant criteria
comparison,,
Riverside -> North,"annual cost, delivery time, capacity",closure days
Riverside -> East,"annual cost, capacity","delivery time, closure days"
Riverside -> West,"annual cost, delivery time","capacity, closure days"
North -> Riverside,"delivery time, capacity, closure days",annual cost
North -> East,"annual cost, capacity, closure days",delivery time
North -> West,"annual cost, delivery time, closure days",capacity
East -> Riverside,"delivery time, closure days","annual cost, capacity"
East -> North,delivery time,"annual cost, capacity, closure days"
East -> West,"annual cost, delivery time, closure days",capacity


In [7]:
df_c.style.background_gradient(vmin=0, vmax=1)

,Riverside,North,East,West
Riverside,1.000000,0.700000,0.500000,0.450000
North,0.750000,1.000000,0.800000,0.750000
East,0.500000,0.200000,1.000000,0.750000
West,0.550000,0.250000,0.250000,1.000000


### 2.4. Discordance index
---
Measure the **largest disadvantage** opposing the assertion $aSb$:

$$ \large
d(a,b)=\max\left(0,\max_j\left[g_j(b)-g_j(a)\right]\right)
$$

The value is zero when no criterion opposes the assertion. Because the scales share $[0,1]$, this index also lies in $[0,1]$. Criterion weights are used in concordance, not in this discordance calculation.

In [8]:
df_d = pd.DataFrame(0.0, index=df.index, columns=df.index)

for a in df.index:
    for b in df.index:
        disadvantages = (df_n.loc[b] - df_n.loc[a]).clip(lower=0)
        df_d.loc[a, b] = disadvantages.max()

df_d.style.background_gradient(vmin=0, vmax=1)

,Riverside,North,East,West
Riverside,0.000000,0.800000,0.700000,0.600000
North,0.150000,0.000000,0.175000,0.250000
East,0.275000,0.166667,0.000000,0.416667
West,0.350000,0.200000,0.350000,0.000000


### 2.5. Concordance and discordance tests
---
Apply the team's two policy parameters:

- **Concordance threshold** $c^*=0.70$: supporting criteria must represent at least 70% of the total importance.
- **Discordance threshold** $d^*=0.35$: no disadvantage may exceed 35% of a criterion's planning range.

For example, $d^*$ allows a disadvantage of at most **7 closure days** or **BRL 1.4 million/year**, using the stated ranges. These are illustrative preferences, not universal ELECTRE constants. Keep self-comparisons out of the relation. A numerical tolerance protects equality at the thresholds from floating-point rounding.

In [9]:
df_c_test = df_c >= C_THRESHOLD - TOLERANCE
df_d_test = df_d <= D_THRESHOLD + TOLERANCE

for i in range(min(df_c_test.shape)):
    df_c_test.iloc[i, i] = False

# Altera a diagonal de df_d_test diretamente no objeto original
for i in range(min(df_d_test.shape)):
    df_d_test.iloc[i, i] = False

pd.concat({
    "concordance test": df_c_test.astype(int),
    "discordance test": df_d_test.astype(int)
}, axis=1)

concordance test                 discordance test                
                 Riverside North East West        Riverside North East West
Riverside                0     1    0    0                0     0    0    0
North                    1     0    1    1                1     0    1    1
East                     0     0    0    1                1     1    0    0
West                     0     0    0    0                1     1    1    0

### 2.6. Outranking relation
---
Accept the assertion only when **both tests** pass:

$$ \large
aSb \iff c(a,b)\geq c^* \quad\text{and}\quad d(a,b)\leq d^*, \qquad a\ne b
$$

An entry 1 represents an arrow from the row alternative to the column alternative. A failed assertion does not, by itself, establish the reverse assertion.

In [10]:
df_s = df_c_test & df_d_test

df_s.astype(int)

,Riverside,North,East,West
Riverside,0,0,0,0
North,1,0,1,1
East,0,0,0,0
West,0,0,0,0


In [11]:
df_edges = pd.DataFrame([
    (a, b) for a in df.index for b in df.index if df_s.loc[a, b]
], columns=["from", "to"])

df_edges

,from,to
0,North,Riverside
1,North,East
2,North,West


In [12]:
df_comparison = pd.DataFrame({
    "North performance": df_n.loc["North"],
    "Riverside performance": df_n.loc["Riverside"],
    "criterion weight": WEIGHTS
})

print(f'Riverside -> North: c = {df_c.loc["Riverside", "North"]:.2f}, '
      f'd = {df_d.loc["Riverside", "North"]:.2f}')
print(f'North -> Riverside: c = {df_c.loc["North", "Riverside"]:.2f}, '
      f'd = {df_d.loc["North", "Riverside"]:.2f}')

df_comparison

Riverside -> North: c = 0.70, d = 0.80
North -> Riverside: c = 0.75, d = 0.15


,North performance,Riverside performance,criterion weight
annual cost,0.650,0.800,0.25
delivery time,0.625,0.625,0.20
capacity,0.500,0.500,0.25
closure days,0.900,0.100,0.30


**Riverside** is cheaper than **North** and matches its delivery time and capacity, giving $c=0.70$. However, its additional 16 closure days give $d=0.80$, blocking the assertion. In the reverse direction, North's annual cost disadvantage is only $0.15$ of the planning range, so both tests pass.

### 2.7. Kernel and choice
---
Find a subset $K$ satisfying:

- **Internal independence**: no two distinct members of $K$ outrank one another.
- **External dominance**: every alternative outside $K$ is outranked by at least one member of $K$.

For these small examples, enumerate the subsets and check both conditions directly. This is not simply a count of outgoing arrows. Acyclic relations have a unique kernel; cycles can require additional treatment and can lead to multiple or no kernels under these conditions.

In [13]:
def find_kernels(relation):
    labels = list(relation.index)
    kernels = []

    for size in range(1, len(labels) + 1):
        for subset in combinations(labels, size):
            members = list(subset)
            others = [label for label in labels if label not in members]
            independent = not relation.loc[members, members].to_numpy().any()
            dominating = relation.loc[members, others].any(axis=0).all()

            if independent and dominating:
                kernels.append(members)

    return kernels


KERNELS = find_kernels(df_s)

KERNELS

[['North']]

In [14]:
assert len(KERNELS) == 1
KERNEL = KERNELS[0]

df_choice = pd.DataFrame({
    "in kernel": [a in KERNEL for a in df.index],
    "outranked by kernel": [
        ", ".join(b for b in KERNEL if df_s.loc[b, a])
        for a in df.index
    ]
}, index=df.index)

df_choice

,in kernel,outranked by kernel
Riverside,False,North
North,True,
East,False,North
West,False,North


**North** is the only member of the kernel and outranks all three other sites. It offers a clear compromise: Riverside's lower cost does not justify its disruption exposure; East's faster delivery and West's larger capacity do not overturn the coalition supporting North. The result selects North without imposing a ranking on the excluded sites.

### 2.8. Threshold sensitivity
---
Repeat the choice for nearby policy thresholds. Stability in this grid supports the interpretation locally; it does not establish robustness to every possible preference or performance change.

In [15]:
sensitivity = []

for c_threshold in [0.65, 0.70, 0.75]:
    for d_threshold in [0.30, 0.35, 0.40]:
        relation = (
            (df_c >= c_threshold - TOLERANCE)
            & (df_d <= d_threshold + TOLERANCE)
        )
        for i in range(min(relation.shape)):
            relation.iloc[i, i] = False
        kernels = find_kernels(relation)
        sensitivity.append({
            "concordance threshold": c_threshold,
            "discordance threshold": d_threshold,
            "kernels": kernels
        })

df_sensitivity = pd.DataFrame(sensitivity)

assert all(row["kernels"] == [["North"]] for row in sensitivity)

df_sensitivity

,concordance threshold,discordance threshold,kernels
0,0.65,0.30,[[North]]
1,0.65,0.35,[[North]]
2,0.65,0.40,[[North]]
3,0.70,0.30,[[North]]
4,0.70,0.35,[[North]]
5,0.70,0.40,[[North]]
6,0.75,0.30,[[North]]
7,0.75,0.35,[[North]]
8,0.75,0.40,[[North]]


## 3. Examples
---

### 3.1. Industrial pump selection
---
Select a replacement pump for a factory's cooling loop. Three proposals meet the required flow and pressure. Compare **energy use** (MWh/year), **maintenance cost** (BRL thousand/year), **noise** (dBA), and expected **unplanned stoppage** (hours/year), all minimized.

The data, common-scale bounds and preferences are **illustrative**. The factory requires at least **65% concordance** and permits no disadvantage exceeding **30% of the assessment range**. Each proposal has an advantage, so no alternative dominates every other on all criteria.

In [16]:
df_ex1 = pd.DataFrame({
    "energy use": [90, 120, 100],
    "maintenance cost": [18, 14, 27],
    "noise": [68, 74, 64],
    "unplanned stoppage": [8, 12, 6]
}, index=["Pump A", "Pump B", "Pump C"])

df_ex1.style.background_gradient()

,energy use,maintenance cost,noise,unplanned stoppage
Pump A,90,18,68,8
Pump B,120,14,74,12
Pump C,100,27,64,6


In [17]:
# Objectives, weights and common-scale bounds
OBJECTIVES_ex1 = pd.Series("min", index=df_ex1.columns)
WEIGHTS_ex1 = pd.Series([0.35, 0.30, 0.15, 0.20], index=df_ex1.columns)

BOUNDS_ex1 = pd.DataFrame({
    "lower": [60, 10, 60, 0],
    "upper": [160, 30, 80, 20]
}, index=df_ex1.columns)

C_THRESHOLD_ex1 = 0.65
D_THRESHOLD_ex1 = 0.30

# Common-scale performances: every criterion is minimized
assert df_ex1.ge(BOUNDS_ex1["lower"]).all().all()
assert df_ex1.le(BOUNDS_ex1["upper"]).all().all()

df_ex1_n = (
    (BOUNDS_ex1["upper"] - df_ex1)
    /(BOUNDS_ex1["upper"] - BOUNDS_ex1["lower"])
)

pd.DataFrame({
    "objective": OBJECTIVES_ex1,
    "weight": WEIGHTS_ex1,
    "lower": BOUNDS_ex1["lower"],
    "upper": BOUNDS_ex1["upper"]
})

,objective,weight,lower,upper
energy use,min,0.35,60,160
maintenance cost,min,0.30,10,30
noise,min,0.15,60,80
unplanned stoppage,min,0.20,0,20


In [18]:
df_ex1_n.style.background_gradient(vmin=0, vmax=1)

,energy use,maintenance cost,noise,unplanned stoppage
Pump A,0.700000,0.600000,0.600000,0.600000
Pump B,0.400000,0.800000,0.300000,0.400000
Pump C,0.600000,0.150000,0.800000,0.700000


In [19]:
# Concordance and discordance indices
df_ex1_c = pd.DataFrame(0.0, index=df_ex1.index, columns=df_ex1.index)
df_ex1_d = df_ex1_c.copy()

for a in df_ex1.index:
    for b in df_ex1.index:
        support = df_ex1_n.loc[a] >= df_ex1_n.loc[b]
        df_ex1_c.loc[a, b] = WEIGHTS_ex1[support].sum()
        df_ex1_d.loc[a, b] = (
            (df_ex1_n.loc[b] - df_ex1_n.loc[a]).clip(lower=0).max()
        )

pd.concat({"concordance": df_ex1_c, "discordance": df_ex1_d}, axis=1)

concordance               discordance              
            Pump A Pump B Pump C      Pump A Pump B Pump C
Pump A        1.00    0.7   0.65        0.00   0.20    0.2
Pump B        0.30    1.0   0.30        0.30   0.00    0.5
Pump C        0.35    0.7   1.00        0.45   0.65    0.0

In [20]:
# Outranking relation and kernel
df_ex1_s = (
    (df_ex1_c >= C_THRESHOLD_ex1 - TOLERANCE)
    & (df_ex1_d <= D_THRESHOLD_ex1 + TOLERANCE)
)

for i in range(min(df_ex1_s.shape)):
    df_ex1_s.iloc[i, i] = False

KERNELS_ex1 = find_kernels(df_ex1_s)

print(f'Kernels: {KERNELS_ex1}')

df_ex1_s.astype(int)

Kernels: [['Pump A']]


,Pump A,Pump B,Pump C
Pump A,0,1,1
Pump B,0,0,0
Pump C,0,0,0


**Pump A** is the unique kernel. Against Pump B, it has concordance $0.70$ and discordance $0.20$: lower energy use, noise and stoppage outweigh the modest maintenance disadvantage. Against Pump C, its lower energy use and maintenance cost give concordance $0.65$, while its greatest disadvantage is $0.20$ for noise. Both assertions pass the stated thresholds.

## 4. References
---

- Roy, B. (1968). [*Classement et choix en présence de points de vue multiples*](https://doi.org/10.1051/ro/196802V100571).
- Figueira, J., Mousseau, V. and Roy, B. (2005). [*ELECTRE methods*](https://www.lamsade.dauphine.fr/mcda/biblio/PDF/JFVMBR2005.pdf). In *Multiple Criteria Decision Analysis: State of the Art Surveys*, pp. 133–162. Section 3.1.1 describes the common-scale ELECTRE I formulation used here.